# AutoGluon for automatic task
Try to use Autogluon for traditional baseline

In [1]:
import os
import logging

os.chdir('/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML')
os.makedirs('logs', exist_ok=True)

logging.basicConfig(       
    level=logging.INFO,            
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',  
    handlers=[
        logging.FileHandler('logs/AutoGluon.log', mode='w'),  
        logging.StreamHandler()          
    ],
    force=True
)

logger = logging.getLogger()

In [2]:
from autogluon.tabular import TabularDataset, TabularPredictor

data_path = '/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/outputs/ustc-binary/raw'
model_root = '/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/AutogluonModels/ustc-binary/raw'
result_path = '/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/results/ustc-binary/raw'

for id in range(3):
    train_data = TabularDataset(f'{data_path}/train_val_split_{id}/train.csv')
    tuning_data = TabularDataset(f'{data_path}/train_val_split_{id}/val.csv')
    test_data = TabularDataset(f'{data_path}/test.csv')

    model_path = f'{model_root}/split_{id}'
    os.makedirs(model_path, exist_ok=True)

    predictor = TabularPredictor(label='class', path=model_path).fit(train_data = train_data, tuning_data = tuning_data, num_gpus = 0, 
                                                    hyperparameters = {
                                                        'GBM': [{}],
                                                        'NN_TORCH': {},
                                                        'XGB': {},
                                                        'RF': [
                                                            {'criterion': 'gini', 'ag_args': {'name_suffix': 'Gini', 'problem_types': ['binary', 'multiclass']}},
                                                            {'criterion': 'entropy', 'ag_args': {'name_suffix': 'Entr', 'problem_types': ['binary', 'multiclass']}}, 
                                                            {'criterion': 'squared_error', 'ag_args': {'name_suffix': 'MSE', 'problem_types': ['regression', 'quantile']}}]
                                                    })

    os.makedirs(result_path, exist_ok=True)
    predictor.leaderboard(
        test_data,
        extra_metrics=['f1_macro', 'f1_micro', 'precision_macro', 'precision_micro', 'recall_macro', 'recall_micro']
    ).to_csv(f'{result_path}/ustc-binary_{id}.csv', index=False)    

/home/a/miniconda3/envs/shallowml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loaded data from: /home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/outputs/ustc-binary/raw/train_val_split_0/train.csv | Columns = 35 / 35 | Rows = 200000 -> 200000
Loaded data from: /home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/outputs/ustc-binary/raw/train_val_split_0/val.csv | Columns = 35 / 35 | Rows = 24000 -> 24000
Loaded data from: /home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/outputs/ustc-binary/raw/test.csv | Columns = 35 / 35 | Rows = 1218664 -> 1218664
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.20
Operating System:   Linux
Platform Machine:   x86_6

In [4]:
import numpy as np
import pandas as pd
import scipy.stats as stats

def calculate_ci(data):
    mean = np.mean(data)
    std_dev = np.std(data, ddof=1)
    sem = std_dev / np.sqrt(len(data))
    confidence_interval = stats.t.interval(0.95, len(data)-1, loc=mean, scale=sem)

    measurement_result = mean
    margin_of_error = (confidence_interval[1] - confidence_interval[0]) / 2

    return measurement_result * 100, margin_of_error * 100

models = ['RandomForestGini', 'XGBoost', 'LightGBM', 'NeuralNetTorch']
acc = {model: [] for model in models}  # 初始化字典，模型名为键，值为空列表
f1_macro = {model: [] for model in models}   # 同上
f1_micro = {model: [] for model in models}   # 同上
inference_time = {model: [] for model in models}

result_path = '/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/results/ustc-binary/raw'

# 遍历不同的id
for id in range(3):
    # 读取当前id的数据
    df = pd.read_csv(f'{result_path}/ustc-binary_{id}.csv')
    # 遍历每个模型，直接从数据中筛选出对应模型的score_test和f1_macro
    for model in models:
        model_data = df[df['model'] == model]
        acc[model].append(model_data['score_test'].values[0])  # 假设每个模型对应的score_test只有一个值
        f1_macro[model].append(model_data['f1_macro'].values[0])  # 同理，f1_macro也只有一个值
        f1_micro[model].append(model_data['f1_micro'].values[0])
        inference_time[model].append(model_data['pred_time_test'].values[0])

for model in models:
    print(model)

    measurement_result, margin_of_error = calculate_ci(acc[model])
    print(f"Reported Acc Result: {measurement_result:.1f} ± {margin_of_error:.1f} (95% confidence level)")

    measurement_result, margin_of_error = calculate_ci(f1_macro[model])
    print(f"Reported F1-Macro Result: {measurement_result:.1f} ± {margin_of_error:.1f} (95% confidence level)")

    measurement_result, margin_of_error = calculate_ci(f1_micro[model])
    print(f"Reported F1-micro Result: {measurement_result:.1f} ± {margin_of_error:.1f} (95% confidence level)")
    
    measurement_result, margin_of_error = calculate_ci(inference_time[model])
    print(f"Reported Time Result: {measurement_result/100:.1f} ± {margin_of_error/100:.1f} (95% confidence level)")

    print()

RandomForestGini
Reported Acc Result: 83.3 ± 37.9 (95% confidence level)
Reported F1-Macro Result: 82.0 ± 39.6 (95% confidence level)
Reported F1-micro Result: 83.3 ± 37.9 (95% confidence level)
Reported Time Result: 1.3 ± 0.5 (95% confidence level)

XGBoost
Reported Acc Result: 81.7 ± 43.0 (95% confidence level)
Reported F1-Macro Result: 80.4 ± 43.9 (95% confidence level)
Reported F1-micro Result: 81.7 ± 43.0 (95% confidence level)
Reported Time Result: 1.0 ± 0.2 (95% confidence level)

LightGBM
Reported Acc Result: 87.3 ± 28.4 (95% confidence level)
Reported F1-Macro Result: 85.4 ± 33.0 (95% confidence level)
Reported F1-micro Result: 87.3 ± 28.4 (95% confidence level)
Reported Time Result: 0.1 ± 0.0 (95% confidence level)

NeuralNetTorch
Reported Acc Result: 92.2 ± 17.3 (95% confidence level)
Reported F1-Macro Result: 91.4 ± 19.1 (95% confidence level)
Reported F1-micro Result: 92.2 ± 17.3 (95% confidence level)
Reported Time Result: 2.5 ± 1.8 (95% confidence level)



In [5]:
import pandas as pd

result_path = '/home/a/traffic_encryption/Debunk_Traffic_Representation/code/ShallowML/results/ustc-binary/raw'
models = ['RandomForestGini', 'XGBoost', 'LightGBM', 'NeuralNetTorch']

for id in range(3):
    print(f'===== split {id} =====')
    df = pd.read_csv(f'{result_path}/ustc-binary_{id}.csv')
    print(df[['model', 'score_test', 'f1_macro', 'f1_micro', 'pred_time_test']])
    print()

    for model in models:
        model_data = df[df['model'] == model]
        if len(model_data) == 0:
            print(f'{model}: NOT FOUND')
        else:
            print(model, 
                  model_data['score_test'].values[0],
                  model_data['f1_macro'].values[0],
                  model_data['f1_micro'].values[0],
                  model_data['pred_time_test'].values[0])
    print()

===== split 0 =====
                 model  score_test  f1_macro  f1_micro  pred_time_test
0     RandomForestEntr    0.999998  0.999997  0.999998        1.354894
1     RandomForestGini    0.999998  0.999997  0.999998        1.501975
2  WeightedEnsemble_L2    0.999998  0.999997  0.999998        1.532519
3       NeuralNetTorch    0.999865  0.999860  0.999865        3.198891
4             LightGBM    0.999570  0.999553  0.999570        0.074581
5              XGBoost    0.998319  0.998250  0.998319        1.087935

RandomForestGini 0.99999753828783 0.9999974398610498 0.99999753828783 1.5019750595092771
XGBoost 0.9983186505878568 0.99825039787707 0.9983186505878568 1.087935447692871
LightGBM 0.9995700209409648 0.9995527863842164 0.9995700209409648 0.0745809078216552
NeuralNetTorch 0.9998654264013708 0.9998600431610506 0.9998654264013708 3.1988914012908936

===== split 1 =====
                 model  score_test  f1_macro  f1_micro  pred_time_test
0       NeuralNetTorch    0.897796  0.892199